# Enhanced S3 to COG Converter with Chunked Processing

This notebook converts TIF files from S3 to Cloud Optimized GeoTIFFs (COGs) with:
- **Chunked processing** for memory-efficient handling of large files
- **Automatic AWS credential detection** (no .env file needed)
- **Download caching** to avoid re-downloading large files
- **COG validation** before uploading
- **Memory monitoring** and progress tracking

Author: Kyle Lesinger (Enhanced chunked version)

In [1]:
import os
import pandas as pd
import json
import tempfile
import boto3
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject
from rasterio.io import MemoryFile
import rioxarray as rxr
import s3fs
import fsspec
from botocore.exceptions import NoCredentialsError, ClientError
from pathlib import Path
from datetime import datetime
import time
import numpy as np
import gc
import psutil
from tqdm import tqdm

print("✅ Libraries imported successfully!")
print(f"Boto3 version: {boto3.__version__}")
print(f"Rasterio version: {rasterio.__version__}")


✅ Libraries imported successfully!
Boto3 version: 1.39.11
Rasterio version: 1.4.3


In [2]:
# Add path for importing custom modules
import sys
from pathlib import Path

# Add the scripts directory to the Python path
scripts_dir = Path('../scripts').resolve()
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

# Import functions from list_s3crawler_files module
from list_s3crawler_files import (
    load_drcs_data,
    get_tif_files_from_path,
    get_files_with_full_paths,
    list_available_directories
)

# Import COG and cache utilities
from cog_utilities import (
    check_cache_status,
    clear_cache,
    validate_cog,
    export_COG_PROFILE
)

# Import AWS S3 utilities
from aws_s3_utils import (
    initialize_s3_client,
    verify_s3_client,
    get_all_s3_keys
)

# Import batch processing utilities
from batch_processing import (
    process_file_batch,
    print_batch_summary
)

from memory_utils import (
    get_memory_usage,
    calculate_optimal_chunk_size,
    estimate_chunk_memory,
    format_bytes

)

from convert_utilities import (
    convert_to_proper_CRS_and_cogify_chunked
)
    
print("✅ Custom modules imported successfully!")
print(f"   Module path: {scripts_dir}")

✅ Memory monitoring utilities loaded
✅ Custom modules imported successfully!
   Module path: /home/jovyan/conversion_scripts/convert-files-and-move/scripts


# Useful links
<a href="https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations/" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">drcs_activations OLD Directory</a> -- You can view old directory file structure here.

<a href="https://docs.openveda.cloud/user-guide/content-curation/dataset-ingestion/file-preparation.html" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">VEDA docs for file naming conventions</a> -- Helps for understanding why/how we name content.

## List of new 2nd level directories

    "Sentinel-1"
    "Sentinel-2"
    "Landsat"
    "MODIS"
    "VIIRS"
    "ASTER"
    "MASTER"
    "ECOSTRESS"
    "Planet"
    "Maxar"
    "HLS"
    "IMERG"
    "GOES"
    "SMAP"
    "ICESat"
    "GEDI"
    "COMSAR"
    "UAVSAR"
    "WB-57"

In [3]:
# DO NOT CHANGE
DIR_OLD_BASE = 'drcs_activations'
DIR_NEW_BASE = 'drcs_activations_new'
BUCKET = 'nasa-disasters'

In [4]:

EVENT_NAME = '202409_Hurricane_Helene'  #find the name within drcs_activations OLD Directory (see link above)
PRODUCT_NAME = 'blackmarble'      #find the name within drcs_activations OLD Directory (see link above)
PATH_OLD = f'{DIR_OLD_BASE}/{EVENT_NAME}/{PRODUCT_NAME}'  # Updated to use actual available directory

In [5]:
# Define COG profile for rasterio (DO NOT CHANGE)
COG_PROFILE = export_COG_PROFILE()

# Chunked processing configuration
CHUNK_CONFIG = {
    "default_chunk_size": 1024,  # Default chunk size in pixels
    "memory_limit_mb": 500,      # Memory limit per chunk in MB
    "show_progress": True,       # Show progress bars
    "enable_memory_monitoring": True  # Monitor memory usage
}

## Initialize AWS S3 Client with automatic credential detection

In [6]:
# Initialize AWS S3 Client using the imported function
s3_client, fs_read = initialize_s3_client(bucket_name=BUCKET, verbose=True)

# Verify S3 client is ready using the imported function
verify_s3_client(s3_client, bucket_name=BUCKET, verbose=True)

# Get all TIF files using the imported function
keys = get_all_s3_keys(s3_client, BUCKET, PATH_OLD, ".tif") if s3_client else []

if keys:
    print(f"✅ Found {len(keys)} .tif files in the S3 bucket.")
else:
    print("No keys found or S3 client not initialized")
    
keys

✅ S3 client initialized successfully
   Found 68 accessible buckets
✅ S3 filesystem (fsspec) initialized
✅ S3 client ready for operations
   Bucket: nasa-disasters
   Ready to process files
✅ Found 101 .tif files in the S3 bucket.


['drcs_activations/202409_Hurricane_Helene/blackmarble/VNP46A2.A2024262.DNB_BRDF-Corrected_NTLC2.Mosaic_C2.tif',
 'drcs_activations/202409_Hurricane_Helene/blackmarble/VNP46A2.A2024262.QF_Cloud_MaskC2_Cloud_C2.Mosaic_C2.tif',
 'drcs_activations/202409_Hurricane_Helene/blackmarble/VNP46A2.A2024263.DNB_BRDF-Corrected_NTLC2.Mosaic_C2.tif',
 'drcs_activations/202409_Hurricane_Helene/blackmarble/VNP46A2.A2024263.QF_Cloud_MaskC2_Cloud_C2.Mosaic_C2.tif',
 'drcs_activations/202409_Hurricane_Helene/blackmarble/VNP46A2.A2024264.DNB_BRDF-Corrected_NTLC2.Mosaic_C2.tif',
 'drcs_activations/202409_Hurricane_Helene/blackmarble/VNP46A2.A2024264.QF_Cloud_MaskC2_Cloud_C2.Mosaic_C2.tif',
 'drcs_activations/202409_Hurricane_Helene/blackmarble/VNP46A2.A2024265.DNB_BRDF-Corrected_NTLC2.Mosaic_C2.tif',
 'drcs_activations/202409_Hurricane_Helene/blackmarble/VNP46A2.A2024265.QF_Cloud_MaskC2_Cloud_C2.Mosaic_C2.tif',
 'drcs_activations/202409_Hurricane_Helene/blackmarble/VNP46A2.A2024266.DNB_BRDF-Corrected_NTLC2

## Configure bucket and paths (no need to create session manually)

In [7]:
def return_bucket_info(config):
    """
    Extract bucket information from configuration and return as dictionary.
    
    Args:
        config: Configuration dictionary containing bucket and prefix information
    
    Returns:
        Dictionary with bucket and prefix information
    """
    # Configure bucket and paths (no need to create session manually)
    bucket_name = config["cog_data_bucket"]
    raw_data_bucket = config["raw_data_bucket"]
    raw_data_prefix = config["raw_data_prefix"]
    
    cog_data_bucket = config['cog_data_bucket']
    cog_data_prefix = config["cog_data_prefix"]
    
    print(f"Configuration loaded:")
    print(f"  Source bucket: {raw_data_bucket}")
    print(f"  Source prefix: {raw_data_prefix}")
    print(f"  Target bucket: {cog_data_bucket}")
    print(f"  Target prefix: {cog_data_prefix}")

    return {
        "bucket_name": bucket_name,
        "raw_data_bucket": raw_data_bucket,
        "raw_data_prefix": raw_data_prefix,
        "cog_data_bucket": cog_data_bucket,
        "cog_data_prefix": cog_data_prefix
    }

## Define Chunked COG Conversion Function

This function handles the conversion of files to Cloud Optimized GeoTIFFs with:
- Chunked processing to handle large files
- Memory monitoring
- Progress tracking
- Proper CRS and caching

In [8]:
# Check current cache status using the imported function
check_cache_status()

📊 Cache Status:
  - Directory: data_download/
  - Total files: 3
  - Total size: 1.05 GB

📁 Cached files (first 10):
  - drcs_activations/202409_Hurricane_Helene/sentinel2/S2A_trueColor_20240922_161001_T17RKN.tif (345.1 MB)
  - drcs_activations/202409_Hurricane_Helene/sentinel2/S2A_trueColor_20240922_161001_T17RKP.tif (345.1 MB)
  - drcs_activations/202410_Hurricane_Milton/sentinel2/S2A_MSIL2A_colorInfrared_20241012_161221_T16RGT.tif (380.3 MB)


(3, 1122450497)

In [9]:
import re

def simple_process_files(keys, filter_str, rename_func, target_dir, EVENT_NAME):
    """
    Simple wrapper to process files with minimal code.
    
    Args:
        keys: List of all S3 keys
        filter_str: Can be:
            - String to filter files (e.g. 'S1_WTR')
            - Regex pattern object (e.g. re.compile(r'.*S2A.*mosaic'))
            - Callable function that returns True/False
        rename_func: Your custom rename function
        target_dir: Target directory (e.g. "Sentinel-1/opera_dswx")
        EVENT_NAME: Event name
    
    Returns:
        Processing results DataFrame
    """
    # 1. Filter files based on type of filter_str
    if callable(filter_str):
        # If it's a function
        filtered_files = [i for i in keys if filter_str(i)]
    elif hasattr(filter_str, 'search'):
        # If it's a compiled regex pattern
        filtered_files = [i for i in keys if filter_str.search(i)]
    elif isinstance(filter_str, str) and filter_str.startswith('r"') or filter_str.startswith("r'"):
        # If it's a regex string (e.g., r'pattern')
        pattern = re.compile(filter_str[2:-1])  # Remove r" or r'
        filtered_files = [i for i in keys if pattern.search(i)]
    else:
        # Default: simple string contains
        filtered_files = [i for i in keys if filter_str in i]
    _
    # 2. Test renaming
    print(f"Testing filenames:")
    for f in filtered_files:
        print(f"  {rename_func(f, EVENT_NAME)}")
    
    # 3. Setup config
    config = {
        "data_acquisition_method": "s3",
        "raw_data_bucket": BUCKET,
        "raw_data_prefix": PATH_OLD,
        "cog_data_bucket": BUCKET,
        "cog_data_prefix": f'{DIR_NEW_BASE}/{target_dir}',
        "local_output_dir": f"output/{EVENT_NAME}",
        "transformation": {}
    }
    return_bucket_info(config)
    
    # 4. Process files
    print("\n" + "="*50)
    print("🌊 Processing Files (Chunked)")
    print("="*50)
    
    def chunked_converter(name, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, s3_client, local_output_dir=None):
        return convert_to_proper_CRS_and_cogify_chunked(
            name, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, s3_client, COG_PROFILE,
            local_output_dir, chunk_config=CHUNK_CONFIG
        )

    results = process_file_batch(
        file_list=filtered_files,
        s3_client=s3_client,
        config=config,
        filename_creator_func=rename_func,
        processing_func=chunked_converter,
        event_name=EVENT_NAME,
        save_metadata=True,
        save_csv=True,
        verbose=True,
        BUCKET=BUCKET
    )
    
    print_batch_summary(results)
    return results

# Process files

In [10]:
keys


['drcs_activations/202409_Hurricane_Helene/blackmarble/VNP46A2.A2024262.DNB_BRDF-Corrected_NTLC2.Mosaic_C2.tif',
 'drcs_activations/202409_Hurricane_Helene/blackmarble/VNP46A2.A2024262.QF_Cloud_MaskC2_Cloud_C2.Mosaic_C2.tif',
 'drcs_activations/202409_Hurricane_Helene/blackmarble/VNP46A2.A2024263.DNB_BRDF-Corrected_NTLC2.Mosaic_C2.tif',
 'drcs_activations/202409_Hurricane_Helene/blackmarble/VNP46A2.A2024263.QF_Cloud_MaskC2_Cloud_C2.Mosaic_C2.tif',
 'drcs_activations/202409_Hurricane_Helene/blackmarble/VNP46A2.A2024264.DNB_BRDF-Corrected_NTLC2.Mosaic_C2.tif',
 'drcs_activations/202409_Hurricane_Helene/blackmarble/VNP46A2.A2024264.QF_Cloud_MaskC2_Cloud_C2.Mosaic_C2.tif',
 'drcs_activations/202409_Hurricane_Helene/blackmarble/VNP46A2.A2024265.DNB_BRDF-Corrected_NTLC2.Mosaic_C2.tif',
 'drcs_activations/202409_Hurricane_Helene/blackmarble/VNP46A2.A2024265.QF_Cloud_MaskC2_Cloud_C2.Mosaic_C2.tif',
 'drcs_activations/202409_Hurricane_Helene/blackmarble/VNP46A2.A2024266.DNB_BRDF-Corrected_NTLC2

In [11]:
def create_cog_filename_blackmarble_simple(f, EVENT_NAME):
    """Create COG filename for blackmarble files with day of year conversion."""
    from datetime import datetime, timedelta
    import re
    from pathlib import Path
    
    filename = Path(f).stem
    extension = Path(f).suffix
    
    # Define month name to number mapping
    month_map = {
        'January': '01', 'February': '02', 'March': '03', 'April': '04',
        'May': '05', 'June': '06', 'July': '07', 'August': '08',
        'September': '09', 'October': '10', 'November': '11', 'December': '12'
    }
    
    # Check if filename already has YYYYMMDD format at the end
    date_at_end = re.search(r'_(\d{8})$', filename)
    if date_at_end:
        # Files like VNP46A2_DNB_BRDF-Corrected_NTLC2_Mosaic_C2_20240918
        date_str = date_at_end.group(1)
        formatted_date = f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:8]}"
        # Remove the date from the end
        base_filename = filename[:date_at_end.start()]
        # Replace dots with underscores
        base_filename = base_filename.replace('.', '_')
        cog_filename = f'{EVENT_NAME}_blackmarble_{base_filename}_{formatted_date}_day{extension}'
    
    # Check for DOY pattern (day of year)
    elif '.A' in filename and re.search(r'\.A(\d{4})(\d{3})', filename):
        # Files like VNP46A2.A2024262.DNB_BRDF-Corrected_NTLC2.Mosaic_C2
        match = re.search(r'\.A(\d{4})(\d{3})', filename)
        year = int(match.group(1))
        doy = int(match.group(2))
        
        # Convert DOY to date
        date = datetime(year, 1, 1) + timedelta(days=doy - 1)
        formatted_date = date.strftime('%Y-%m-%d')
        
        # Replace the .AYYYYDDD part with nothing and replace remaining dots with underscores
        base_filename = re.sub(r'\.A\d{4}\d{3}', '', filename)
        base_filename = base_filename.replace('.', '_')
        cog_filename = f'{EVENT_NAME}_blackmarble_{base_filename}_{formatted_date}_day{extension}'
    
    # Check for monthly pattern
    elif '.A' in filename and re.search(r'\.A(\d{4})\.(\d{2})\.', filename):
        # Files like VNP46A3.A2024.00.August2024.Mosaic_C2
        match = re.search(r'\.A(\d{4})\.(\d{2})\.(\w+)', filename)
        if match:
            year = match.group(1)
            month_name = match.group(3)
            # Extract just the month name without year if it's like "August2024"
            month_only = re.sub(r'\d{4}$', '', month_name)
            
            # Convert month name to number
            if month_only in month_map:
                month_num = month_map[month_only]
                formatted_date = f"{year}-{month_num}"
            else:
                formatted_date = f"{year}-{month_name}"
            
            # Remove the date pattern and month name, then replace dots
            base_filename = re.sub(r'\.A\d{4}\.\d{2}\.\w+', '', filename)
            base_filename = base_filename.replace('.', '_')
            cog_filename = f'{EVENT_NAME}_blackmarble_{base_filename}_{formatted_date}_monthly{extension}'
    
    # Check for MonthlyComposite pattern
    elif 'MonthlyComposite' in filename and re.search(r'A(\d{4})\.(\w+)', filename):
        # Files like VNP46A3_MonthlyComposite.A2024.August.Augusta
        match = re.search(r'A(\d{4})\.(\w+)', filename)
        if match:
            year = match.group(1)
            month_name = match.group(2)
            
            # Convert month name to number
            if month_name in month_map:
                month_num = month_map[month_name]
                formatted_date = f"{year}-{month_num}"
            else:
                formatted_date = f"{year}-{month_name}"
            
            # Remove the date pattern and replace dots
            base_filename = re.sub(r'\.A\d{4}\.\w+(?:\.\w+)?', '', filename)
            base_filename = base_filename.replace('.', '_')
            cog_filename = f'{EVENT_NAME}_blackmarble_{base_filename}_{formatted_date}_monthly{extension}'
    
    # Check for special Augusta files with dates
    elif 'Augusta' in filename and re.search(r'(\d{8})', filename):
        # Files like VNP46A2_BRDFCorrected_Sep28_Augusta_20240928
        date_match = re.search(r'(\d{8})', filename)
        date_str = date_match.group(1)
        formatted_date = f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:8]}"
        # Remove the YYYYMMDD date and replace dots
        base_filename = re.sub(r'_\d{8}', '', filename)
        base_filename = base_filename.replace('.', '_')
        cog_filename = f'{EVENT_NAME}_blackmarble_{base_filename}_{formatted_date}_day{extension}'
    
    # Check for Augusta files with .AYYYYDDD.MonthDD format
    elif 'Augusta' in filename and re.search(r'\.A(\d{4})(\d{3})\.', filename):
        # Files like VNP46A2_BRDFCorrected.A2024272.Sep28.Augusta
        match = re.search(r'\.A(\d{4})(\d{3})', filename)
        year = int(match.group(1))
        doy = int(match.group(2))
        
        # Convert DOY to date
        date = datetime(year, 1, 1) + timedelta(days=doy - 1)
        formatted_date = date.strftime('%Y-%m-%d')
        
        # Remove the .AYYYYDDD.MonthDD part and replace dots
        base_filename = re.sub(r'\.A\d{4}\d{3}\.\w+\d+', '', filename)
        base_filename = base_filename.replace('.', '_')
        cog_filename = f'{EVENT_NAME}_blackmarble_{base_filename}_Augusta_{formatted_date}_day{extension}'
    
    else:
        # Fallback for any unmatched pattern - still replace dots
        cleaned_filename = filename.replace('.', '_')
        cog_filename = f'{EVENT_NAME}_blackmarble_{cleaned_filename}_day{extension}'
    
    return cog_filename

filter_str = 'blackmarble'

# Test functions
print("Testing WM filename:")
filter_ = [i for i in keys if filter_str in i]
print(len(filter_))

for idx,i in enumerate(filter_):
    test_wm = create_cog_filename_blackmarble_simple(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")



Testing WM filename:
101
  202409_Hurricane_Helene_blackmarble_VNP46A2_DNB_BRDF-Corrected_NTLC2_Mosaic_C2_2024-09-18_day.tif
  202409_Hurricane_Helene_blackmarble_VNP46A2_QF_Cloud_MaskC2_Cloud_C2_Mosaic_C2_2024-09-18_day.tif
  202409_Hurricane_Helene_blackmarble_VNP46A2_DNB_BRDF-Corrected_NTLC2_Mosaic_C2_2024-09-19_day.tif
  202409_Hurricane_Helene_blackmarble_VNP46A2_QF_Cloud_MaskC2_Cloud_C2_Mosaic_C2_2024-09-19_day.tif
  202409_Hurricane_Helene_blackmarble_VNP46A2_DNB_BRDF-Corrected_NTLC2_Mosaic_C2_2024-09-20_day.tif
  202409_Hurricane_Helene_blackmarble_VNP46A2_QF_Cloud_MaskC2_Cloud_C2_Mosaic_C2_2024-09-20_day.tif
  202409_Hurricane_Helene_blackmarble_VNP46A2_DNB_BRDF-Corrected_NTLC2_Mosaic_C2_2024-09-21_day.tif
  202409_Hurricane_Helene_blackmarble_VNP46A2_QF_Cloud_MaskC2_Cloud_C2_Mosaic_C2_2024-09-21_day.tif
  202409_Hurricane_Helene_blackmarble_VNP46A2_DNB_BRDF-Corrected_NTLC2_Mosaic_C2_2024-09-22_day.tif
  202409_Hurricane_Helene_blackmarble_VNP46A2_QF_Cloud_MaskC2_Cloud_C2_Mosa

In [12]:
# Process S1 WTR files
results1 = simple_process_files(keys=keys, 
                                filter_str = filter_str, 
                                rename_func = create_cog_filename_blackmarble_simple, 
                                target_dir = "Blackmarble", 
                                EVENT_NAME = EVENT_NAME)


Testing filenames:
  202409_Hurricane_Helene_blackmarble_VNP46A2_DNB_BRDF-Corrected_NTLC2_Mosaic_C2_2024-09-18_day.tif
  202409_Hurricane_Helene_blackmarble_VNP46A2_QF_Cloud_MaskC2_Cloud_C2_Mosaic_C2_2024-09-18_day.tif
  202409_Hurricane_Helene_blackmarble_VNP46A2_DNB_BRDF-Corrected_NTLC2_Mosaic_C2_2024-09-19_day.tif
  202409_Hurricane_Helene_blackmarble_VNP46A2_QF_Cloud_MaskC2_Cloud_C2_Mosaic_C2_2024-09-19_day.tif
  202409_Hurricane_Helene_blackmarble_VNP46A2_DNB_BRDF-Corrected_NTLC2_Mosaic_C2_2024-09-20_day.tif
  202409_Hurricane_Helene_blackmarble_VNP46A2_QF_Cloud_MaskC2_Cloud_C2_Mosaic_C2_2024-09-20_day.tif
  202409_Hurricane_Helene_blackmarble_VNP46A2_DNB_BRDF-Corrected_NTLC2_Mosaic_C2_2024-09-21_day.tif
  202409_Hurricane_Helene_blackmarble_VNP46A2_QF_Cloud_MaskC2_Cloud_C2_Mosaic_C2_2024-09-21_day.tif
  202409_Hurricane_Helene_blackmarble_VNP46A2_DNB_BRDF-Corrected_NTLC2_Mosaic_C2_2024-09-22_day.tif
  202409_Hurricane_Helene_blackmarble_VNP46A2_QF_Cloud_MaskC2_Cloud_C2_Mosaic_C2_

Reading input: /tmp/tmp352i0ps2_temp.tif



   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmpk8mr625w.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Blackmarble/202409_Hurricane_Helene_blackmarble_VNP46A2_DNB_BRDF-Corrected_NTLC2_Mosaic_C2_2024-09-18_day.tif
   [MEMORY] Final: 512.5 MB (Change: +217.3 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202409_Hurricane_Helene_blackmarble_VNP46A2_DNB_BRDF-Corrected_NTLC2_Mosaic_C2_2024-09-18_day.tif

[2/101] Processing: drcs_activations/202409_Hurricane_Helene/blackmarble/VNP46A2.A2024262.QF_Cloud_MaskC2_Cloud_C2.Mosaic_C2.tif
   Output filename: 202409_Hurricane_Helene_blackmarble_VNP46A2_QF_Cloud_MaskC2_Cloud_C2_Mosaic_C2_2024-09-18_day.tif
   [MEMORY] Initial: 508.7 MB
   [DOWNLOAD] Downloading from S3...


...

   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Blackmarble/202409_Hurricane_Helene_blackmarble_VNP46A2_BRDFCorrected_Sep28_Augusta_2024-09-28_day.tif
   [MEMORY] Final: 638.1 MB (Change: +105.4 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202409_Hurricane_Helene_blackmarble_VNP46A2_BRDFCorrected_Sep28_Augusta_2024-09-28_day.tif

[101/101] Processing: drcs_activations/202409_Hurricane_Helene/blackmarble/blackmarble_hd/VNP46A3_MonthlyComposite.A2024.August.Augusta.tif
   Output filename: 202409_Hurricane_Helene_blackmarble_VNP46A3_MonthlyComposite_2024-08_monthly.tif
   [MEMORY] Initial: 532.7 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Already in EPSG:4326, skipping reprojection
   [VERIFY] Checking 

Reading input: /tmp/tmpcw96y2uk_temp.tif



   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmpkimyqb3i.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Blackmarble/202409_Hurricane_Helene_blackmarble_VNP46A3_MonthlyComposite_2024-08_monthly.tif
   [MEMORY] Final: 823.5 MB (Change: +290.8 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202409_Hurricane_Helene_blackmarble_VNP46A3_MonthlyComposite_2024-08_monthly.tif

✅ Batch processing complete: 101 files processed
📊 Uploaded metadata to s3://nasa-disasters/drcs_activations_new/Blackmarble/metadata.json
📝 Saved processing log to s3://nasa-disasters/drcs_activations_new/Blackmarble/files_converted.csv
📁 COGs saved locally to: output/202409_Hurricane_Helene

📊 BATCH PROCESSING SUMMARY
Total files processed: 101
Successful: 101
Failed: 0
Success rate: 100.0%
Timestamp: 2025-09-16T01:54:12.999178


## Check STATUS of file conversion and upload

<a href="https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations_new/" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">Disasters Bucket</a> -- You can view that the files actually made it to their correct destination.

## Memory Usage Summary

You can check the final memory usage and cleanup

In [ ]:
# Final memory cleanup and report
gc.collect()
final_memory = get_memory_usage()
print(f"\n📊 Memory Usage Summary:")
print(f"  Current memory usage: {final_memory:.1f} MB")
print(f"  Available memory: {psutil.virtual_memory().available / 1024 / 1024:.1f} MB")
print(f"  Memory percent used: {psutil.virtual_memory().percent:.1f}%")